In [1]:
import mteb
import pandas as pd
import os
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv()

/home/sagemaker-user/bertimbau-sentence-transformer/.venv_st/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
tasks = mteb.get_tasks(languages=["por"], modalities=['text'])

for task in tasks:
    print(task)

BibleNLPBitextMining(name='BibleNLPBitextMining', languages=['eng', 'por'])
FloresBitextMining(name='FloresBitextMining', languages=['ace', 'acm', 'acq', '...'])
NTREXBitextMining(name='NTREXBitextMining', languages=['arb', 'ben', 'cat', '...'])
TatoebaBitextMining(name='Tatoeba', languages=['eng', 'por'])
WebFAQBitextMiningQAs(name='WebFAQBitextMiningQAs', languages=['cat', 'dan', 'deu', '...'])
WebFAQBitextMiningQuestions(name='WebFAQBitextMiningQuestions', languages=['cat', 'dan', 'deu', '...'])
AfriSentiClassification(name='AfriSentiClassification', languages=['por'])
AfriSentiLangClassification(name='AfriSentiLangClassification', languages=['amh', 'arq', 'ary', '...'])
LanguageClassification(name='LanguageClassification', languages=['ara', 'bul', 'cmn', '...'])
MassiveIntentClassification(name='MassiveIntentClassification', languages=['por'])
MassiveScenarioClassification(name='MassiveScenarioClassification', languages=['por'])
MultiHateClassification(name='MultiHateClassification

In [3]:
model_name = "Qwen/Qwen3-Embedding-0.6B"
tasks = [
    ("Assin2STS", None),
    ("SICK-BR-STS", None),
    ("STSBenchmarkMultilingualSTS", 'pt'),
    
    ('MassiveIntentClassification', 'pt'),
    ('MultiHateClassification', 'por'),
    ('BrazilianToxicTweetsClassification', None),
    ('HateSpeechPortugueseClassification', None),
    ('TweetSentimentClassification', 'portuguese'),

    ('MultiLongDocReranking', 'pt'),
    ('WikipediaRerankingMultilingual', 'pt'),
    ('XGlueWPRReranking', 'pt'),

    ('WebFAQRetrieval', 'por'),
    ('MultiLongDocRetrieval', 'pt'),
    ('WikipediaRetrievalMultilingual', 'pt')
]

In [4]:
model_meta = mteb.get_model(model_name, device="cuda")
# model_meta = SentenceTransformer(model_name, device="cuda")

# select the desired tasks and evaluate
task_name_list = []
model_name_list = []
main_score_list = []

for task_info in tasks:
    print(f"""
    #############################

    Avaliando {task_info[0]} ({task_info[1]})...
    
    #############################
    """)

    task = mteb.get_task(task_info[0], languages=['por'], hf_subsets=task_info[1])

    # with encode kwargs
    result = mteb.evaluate(model_meta, task, encode_kwargs={"batch_size": 2})

    task_name = result.task_results[0].task_name
    model_name = result.model_name
    main_score = result.task_results[0].main_score

    task_name_list.append(task_name)
    model_name_list.append(model_name)
    main_score_list.append(main_score)

print("Avaliação concluída!")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 5441.47it/s]



    #############################

    Avaliando Assin2STS (None)...

    #############################
    

    #############################

    Avaliando SICK-BR-STS (None)...

    #############################
    

    #############################

    Avaliando STSBenchmarkMultilingualSTS (pt)...

    #############################
    

    #############################

    Avaliando MassiveIntentClassification (pt)...

    #############################
    

    #############################

    Avaliando MultiHateClassification (por)...

    #############################
    

    #############################

    Avaliando BrazilianToxicTweetsClassification (None)...

    #############################
    

    #############################

    Avaliando HateSpeechPortugueseClassification (None)...

    #############################
    

    #############################

    Avaliando TweetSentimentClassification (portuguese)...

    #############################
   

Prompt type 'PromptType.document' not found in task metadata for task 'MultiLongDocReranking'.
/home/sagemaker-user/bertimbau-sentence-transformer/.venv_st/lib/python3.11/site-packages/mteb/models/abs_encoder.py:221: UserWarning: Prompt type 'PromptType.document' not found in task metadata for task 'MultiLongDocReranking'.
  warnings.warn(msg)



    #############################

    Avaliando WikipediaRerankingMultilingual (pt)...

    #############################
    

    #############################

    Avaliando XGlueWPRReranking (pt)...

    #############################
    

    #############################

    Avaliando WebFAQRetrieval (por)...

    #############################
    


In [1]:
df_results = pd.DataFrame({
    'model_name': model_name_list,
    'task_name': task_name_list,
    'main_score': main_score_list
})

NameError: name 'pd' is not defined

In [ ]:
filepath = '../data/results_eval_mteb.csv'

if os.path.exists(filepath):
    df_results_cache = pd.read_csv(filepath)
    df_results = pd.concat([df_results_cache, df_results], axis=0, ignore_index=True)

df_results.to_csv(filepath, index=False)